In [129]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
from yellowbrick.cluster import KElbowVisualizer
import matplotlib.pyplot as plt
import pandas as pd 
import seaborn as sns
from sklearn.cluster import KMeans
from sksurv.base import SurvivalAnalysisMixin as s
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score
from sksurv.preprocessing import encode_categorical
from sksurv.datasets import load_gbsg2
from sksurv.functions import StepFunction
from sksurv.linear_model import CoxPHSurvivalAnalysis, CoxnetSurvivalAnalysis
from sksurv.ensemble import (ComponentwiseGradientBoostingSurvivalAnalysis, 
                            RandomSurvivalForest, 
                            ExtraSurvivalTrees, 
                            GradientBoostingSurvivalAnalysis, 
                            ExtraSurvivalTrees)
from sksurv.meta import EnsembleSelection, EnsembleSelectionRegressor
from sksurv.metrics import integrated_brier_score
from matplotlib.colors import ListedColormap
from mlxtend.evaluate import paired_ttest_5x2cv
from mlxtend.evaluate import combined_ftest_5x2cv
from lifelines import KaplanMeierFitter
from scipy.cluster import hierarchy
from lifelines.statistics import logrank_test, multivariate_logrank_test, pairwise_logrank_test
from sklearn import preprocessing
from sklearn.model_selection import StratifiedKFold, KFold
from lifelines.plotting import add_at_risk_counts
import scipy.stats
import sklearn
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import optuna
from sklearn.model_selection import cross_val_score
from sksurv.metrics import integrated_brier_score
from lifelines import CoxPHFitter
from lifelines.statistics import proportional_hazard_test
import scipy.stats as stats
from statsmodels.stats.outliers_influence import variance_inflation_factor 
 
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'
from sklearn.preprocessing import PowerTransformer


In [130]:
# OUS: Train data
OUS_D1 = pd.read_csv('OUS_D1.csv')
OUS_D2 = pd.read_csv('OUS_D2.csv')
OUS_D3 = pd.read_csv('OUS_D3.csv')
OUS_DFS_target = pd.read_csv('OUS_DFS_target.csv')
OUS_OS_target = pd.read_csv('OUS_OS_target.csv')
response_OUS = pd.read_csv('response_ous.csv', sep=';')

# MAASTRO: Test data 
MAASTRO_D1 = pd.read_csv('MAASTRO_D1.csv')
MAASTRO_D2 = pd.read_csv('MAASTRO_D2.csv')
MAASTRO_D3 = pd.read_csv('MAASTRO_D3.csv')
MAASTRO_DFS_target = pd.read_csv('MAASTRO_DFS_target.csv')
MAASTRO_OS_target = pd.read_csv('MAASTRO_OS_target.csv')
response_MAASTRO = pd.read_csv('maastro_response_full.csv', sep=',')

In [131]:
# Need to choose patient_id from OUS_D1 in response_OUS
data = list(OUS_D1['patient_id'])
mask = response_OUS['patient_id'].isin(data)
response_OUS = response_OUS[mask] 

# Merge OUS_D2 with response_OUS
clinical_train = pd.merge(OUS_D1, response_OUS, on='patient_id', how='inner')
clinical_train = clinical_train.loc[:, ~clinical_train.columns.isin(['DFS', 'event_DFS', 'LRC', 'event_LRC'])]

In [132]:
# Drop patient_id column
clinical_train = clinical_train.drop('patient_id', axis=1)

In [133]:
# Check null values in D1
clinical_train.isnull().sum().sum()

0

In [134]:
""" Takes too long time 
# Check collinearity for OUS data 
df = clinical_train

# Create a correlation matrix
correlation_matrix = df.corr()

plt.figure(figsize=(15, 12))

# Plot a heatmap
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', linewidths=0.5)
plt.show()
""" 

" Takes too long time \n# Check collinearity for OUS data \ndf = clinical_train\n\n# Create a correlation matrix\ncorrelation_matrix = df.corr()\n\nplt.figure(figsize=(15, 12))\n\n# Plot a heatmap\nsns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', linewidths=0.5)\nplt.show()\n"

## Test dataset: MAASTRO 

In [135]:
(MAASTRO_D1['patient_id'] == MAASTRO_OS_target['patient_id']).sum()

99

In [136]:
# Rename the column name of response_MAASTRO 
response_MAASTRO.rename(columns = {'Index' : 'patient_id'}, inplace = True)

In [137]:
# need to choose patient_id from MAASTRO_D2 in response_MAASTRO
data = list(MAASTRO_D1['patient_id'])
mask = response_MAASTRO['patient_id'].isin(data)
response_MAASTRO = response_MAASTRO[mask] 
response_MAASTRO

,patient_id,OS,OS_event,LRC,LRC_event,DFS,DFS_event
0,1,62.43,0.0,62.43,0.0,62.43,0.0
1,2,60.00,0.0,60.00,0.0,60.00,0.0
2,3,44.43,1.0,8.83,1.0,8.83,1.0
3,4,37.20,1.0,19.37,0.0,19.73,1.0
5,6,59.23,0.0,59.23,0.0,59.23,0.0
...,...,...,...,...,...,...,...
109,110,19.00,1.0,13.27,1.0,13.27,1.0
110,111,85.87,1.0,82.83,0.0,85.87,1.0
111,112,42.87,0.0,42.87,0.0,42.87,0.0
112,113,58.93,0.0,58.93,0.0,58.93,0.0


In [138]:
# Merge MAASTRO_D1 with response_MAASTRO
clinical_test = pd.merge(MAASTRO_D1, response_MAASTRO, on='patient_id', how='inner')
clinical_test = clinical_test.loc[:, ~clinical_test.columns.isin(['DFS', 'DFS_event', 'LRC', 'LRC_event'])]
clinical_test

,patient_id,age,female,cavum_oris,oropharynx,hypopharynx,larynx,histgrade_high,hpv_related,charlson,pack_years,uicc8_III-IV,SUVpeak,MTV,TLG,OS,OS_event
0,1,55,0,0,1,0,0,1,1,1,0,0,15.438583,22.841,263.611623,62.43,0.0
1,2,55,0,0,1,0,0,0,0,0,20,1,8.829353,5.660,36.980700,60.00,0.0
2,3,55,0,0,1,0,0,0,0,1,6,1,13.476123,7.791,74.636342,44.43,1.0
3,4,61,1,0,0,0,1,1,0,1,45,1,8.632732,7.908,46.791979,37.20,1.0
4,6,70,0,0,1,0,0,1,1,1,59,0,9.783954,15.237,107.637514,59.23,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94,110,66,1,0,0,0,1,0,0,0,55,1,31.338410,6.110,144.490782,19.00,1.0
95,111,63,0,0,0,0,1,0,0,1,174,1,13.041604,7.182,69.214868,85.87,1.0
96,112,63,0,0,1,0,0,1,1,1,0,1,8.944517,16.483,102.594274,42.87,0.0
97,113,54,0,0,1,0,0,1,1,0,0,0,14.184236,9.981,103.229492,58.93,0.0


In [139]:
# Drop patient_id column
clinical_test = clinical_test.drop('patient_id', axis=1)

In [140]:
# Some rows have null values in OS, OS_event -> Remove those rows
clinical_test[clinical_test.isnull().any(axis=1)]
clinical_test = clinical_test.dropna(how='any',axis=0) 

,age,female,cavum_oris,oropharynx,hypopharynx,larynx,histgrade_high,hpv_related,charlson,pack_years,uicc8_III-IV,SUVpeak,MTV,TLG,OS,OS_event


In [141]:
# X
X = clinical_train.loc[:, ~clinical_train.columns.isin(['OS', 'event_OS'])]

# y 
y = clinical_train.loc[:, ['OS', 'event_OS']]

# Set lower, upper time point and times for IBS calculation later 
lower, upper = np.percentile(y['OS'], [10, 90])
times = np.arange(lower, upper)

In [142]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [143]:
# Shape
print('X_train: ', X.shape)
print('y_train: ', y.shape)

X_train:  (139, 14)
y_train:  (139,)


In [144]:
clinical_test.rename(columns = {'OS_event' : 'event_OS'}, inplace = True)

In [145]:
# X
X_MAASTRO = clinical_test.loc[:, ~clinical_test.columns.isin(['OS', 'event_OS'])]

# y y_MAASTRO
y_MAASTRO = clinical_test.loc[:, ['OS', 'event_OS']]
lower, upper = np.percentile(y_MAASTRO['OS'], [10, 90])
times = np.arange(lower, upper)

# y into array 
lists = [] 
for i, j in zip(y_MAASTRO['event_OS'], y_MAASTRO['OS']): 
    lists.append((i, j))

y_MAASTRO = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

clinical_test.shape

(99, 16)

# Feature Selection: RENT

In [146]:
# Choose features from the result of RENT 
selected_features = ["age",
"cavum_oris",
"hpv_related",
"uicc8_III-IV",
"oropharynx",
"pack_years",
"charlson"]

In [147]:
X_rent = X.loc[:, selected_features]
X_new = X_rent.copy()

In [148]:
# Selecct the columns from X_MAASTRO
MAASTRO_new = X_MAASTRO.loc[:, selected_features]

# Yeo-Johnson Transformation

In [149]:
# Copy the original X for later 
original_X = X.copy()

In [150]:
X_new.columns

Index(['age', 'cavum_oris', 'hpv_related', 'uicc8_III-IV', 'oropharynx',
       'pack_years', 'charlson'],
      dtype='object')

In [151]:
# Transform X_new 
# Set the categorical_columns
categorical_columns = ['female', 
                        'cavum_oris',
                        'oropharynx',
                        'hypopharynx',
                        'larynx',
                        'histgrade_high',
                        'hpv_related',
                        'charlson',
                        'uicc8_III-IV']

# Set the columns_to_drop which are categorical 
columns_to_drop = [col for col in X_new.columns if col in categorical_columns]

# Drop the columns if they exist in the DataFrame and save the numeric part in X_new_numeric
X_new_categoric = X_new[columns_to_drop]
X_new_numeric = X_new.drop(columns=columns_to_drop, inplace=False)

# Apply Yeo-Johnson transformation
pt = PowerTransformer(method='yeo-johnson')
X_new_numeric_transformed = pt.fit_transform(X_new_numeric)

# Create DataFrame with transformed numerical data
X_new_numeric_transformed = pd.DataFrame(X_new_numeric_transformed, 
                                         columns=X_new_numeric.columns, 
                                         index=X_new.index)

# Concatenate transformed numerical data with categorical data
X_new_std = pd.concat([X_new_numeric_transformed, X_new_categoric], axis=1)
X_new_std = X_new_std.reindex(columns=X_new.columns)

# Standardize X_MAASTRO 
# Set the columns_to_drop which are categorical 
columns_to_drop = [col for col in MAASTRO_new.columns if col in categorical_columns]

# Drop the columns if they exist in the DataFrame and save the numeric part in X_new_numeric
MAASTRO_new_categoric = MAASTRO_new[columns_to_drop]
MAASTRO_new_numeric = MAASTRO_new.drop(columns=columns_to_drop, inplace=False)

# Do the transformation for the numeric part 
MAASTRO_new_numeric_columns = MAASTRO_new_numeric.columns
MAASTRO_new_numeric_index = MAASTRO_new_numeric.index 
MAASTRO_new_numeric_std = pt.transform(MAASTRO_new_numeric)
MAASTRO_new_numeric_std = pd.DataFrame(MAASTRO_new_numeric_std,
                                 columns=MAASTRO_new_numeric_columns, 
                                 index=MAASTRO_new_numeric_index)
MAASTRO_new_std = pd.concat([MAASTRO_new_numeric_std, MAASTRO_new_categoric], axis=1)

# Change the order of the X_new_std 
MAASTRO_new = MAASTRO_new.reindex(columns=X_new.columns)
MAASTRO_new_std = MAASTRO_new_std.reindex(columns=X_new.columns)

In [152]:
X_new

,age,cavum_oris,hpv_related,uicc8_III-IV,oropharynx,pack_years,charlson
0,54.238356,0,0.0,0.0,1,0.000000,0
1,54.539726,0,0.0,0.0,0,27.404795,1
2,59.019178,1,0.0,1.0,0,41.019178,1
3,70.726027,0,0.0,0.0,0,37.500000,1
4,67.865753,0,0.0,0.0,0,53.000000,1
...,...,...,...,...,...,...,...
134,60.435616,0,1.0,0.0,1,0.000000,0
135,68.794521,0,1.0,1.0,1,0.000000,0
136,57.498630,0,1.0,0.0,1,39.498630,1
137,65.684932,0,1.0,1.0,1,71.527397,1


In [153]:
X_new_std

,age,cavum_oris,hpv_related,uicc8_III-IV,oropharynx,pack_years,charlson
0,-0.785036,0,0.0,0.0,1,-1.533345,0
1,-0.746998,0,0.0,0.0,0,0.397233,1
2,-0.175256,1,0.0,1.0,0,0.830437,1
3,1.371631,0,0.0,0.0,0,0.727938,1
4,0.987008,0,0.0,0.0,0,1.144253,1
...,...,...,...,...,...,...,...
134,0.007949,0,1.0,0.0,1,-1.533345,0
135,1.111443,0,1.0,1.0,1,-1.533345,0
136,-0.370651,0,1.0,0.0,1,0.786825,1
137,0.696583,0,1.0,1.0,1,1.554125,1


In [154]:
MAASTRO_new 

,age,cavum_oris,hpv_related,uicc8_III-IV,oropharynx,pack_years,charlson
0,55,0,1,0,1,0,1
1,55,0,0,1,1,20,0
2,55,0,0,1,1,6,1
3,61,0,0,1,0,45,1
4,70,0,1,0,1,59,1
...,...,...,...,...,...,...,...
94,66,0,0,1,0,55,0
95,63,0,0,1,0,174,1
96,63,0,1,1,1,0,1
97,54,0,1,0,1,0,0


In [155]:
MAASTRO_new_std

,age,cavum_oris,hpv_related,uicc8_III-IV,oropharynx,pack_years,charlson
0,-0.688798,0,1,0,1,-1.533345,1
1,-0.688798,0,0,1,1,0.104884,0
2,-0.688798,0,0,1,1,-0.713198,1
3,0.081262,0,0,1,0,0.940197,1
4,1.273609,0,1,0,1,1.285347,1
...,...,...,...,...,...,...,...
94,0.738387,0,0,1,0,1.192311,0
95,0.342484,0,0,1,0,3.094727,1
96,0.342484,0,1,1,1,-1.533345,1
97,-0.815082,0,1,0,1,-1.533345,0


# Modelling 

### 1. CoxPHSurvivalAnalysis

#### Train

In [156]:
# Setting the y format for skf below  
y = clinical_train[['OS', 'event_OS']]

# Running to optuna for hyperparameter tuning 
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class()
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Transformation
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            pt = PowerTransformer(method='yeo-johnson')
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = pt.fit_transform(X_train_included)
                X_test_included_std = pt.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxPHSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxPHSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-13 23:59:31,227] A new study created in memory with name: no-name-c201f636-a6a6-4ce7-80f8-29e6e0548b7c


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.7662337662337663
Fold 2 C-index: 0.7455357142857143
Fold 3 C-index: 0.8186274509803921
Fold 4 C-index: 0.8481012658227848


[I 2024-04-13 23:59:33,613] A new study created in memory with name: no-name-9f7b5ff9-40a9-4a38-8d46-5397f32930f4


Fold 5 C-index: 0.6384976525821596
[I 2024-04-13 23:59:33,604] Trial 0 finished with value: 0.7633991699809635 and parameters: {}. Best is trial 0 with value: 0.7633991699809635.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.7633991699809635], datetime_start=datetime.datetime(2024, 4, 13, 23, 59, 31, 321698), datetime_complete=datetime.datetime(2024, 4, 13, 23, 59, 33, 603104), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.7633991699809635


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.16083908303470473
Fold 2 IBS: 0.17782936802981544
Fold 3 IBS: 0.15392547258993877
Fold 4 IBS: 0.1487529169730995
Fold 5 IBS: 0.2589625484030112
[I 2024-04-13 23:59:35,956] Trial 0 finished with value: 0.1800618778061139 and parameters: {}. Best is trial 0 with value: 0.1800618778061139.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.1800618778061139], datetime_start=datetime.datetime(2024, 4, 13, 23, 59, 33, 971562), datetime_complete=datetime.datetime(2024, 4, 13, 23, 59, 35, 955714), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.1800618778061139


In [157]:
# Setting a dictionary to save the train results 
train_cindex = {} 
train_ibs = {} 

# Saving the values to the dictionary 
train_cindex['CoxPH'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxPH'] = np.round(study_ibs.best_value, 3)

In [158]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.763
train_ibs:  0.18


#### Test

In [159]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [160]:
# Test on MAASTRO 
cph = CoxPHSurvivalAnalysis()

cph.fit(X_new_std, y)

# Save C-index 
c_index = cph.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print('Concordance index:', c_index)

# Save IBS 
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in cph.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print('IBS score:', ibs)

CoxPHSurvivalAnalysis()

Concordance index: 0.572
IBS score: 0.274


In [161]:
# Setting a dictionary to save the test results 
test_cindex = {} 
test_ibs = {} 

In [162]:
# Saving the values to the dictionary 
test_cindex['CoxPH'] = c_index
test_ibs['CoxPH'] = ibs

### 2. CoxnetSurvivalAnalysis

#### Train

In [163]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

# Running to optuna for hyperparameter tuning 
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class(l1_ratio=0.0000001, 
                            fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Transformation
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            pt = PowerTransformer(method='yeo-johnson')
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = pt.fit_transform(X_train_included)
                X_test_included_std = pt.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-13 23:59:36,921] A new study created in memory with name: no-name-0675379c-702b-4d41-8f28-c4ba47533a01


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.6948051948051948
Fold 2 C-index: 0.640625
Fold 3 C-index: 0.7524509803921569
Fold 4 C-index: 0.7784810126582279


[I 2024-04-13 23:59:38,589] A new study created in memory with name: no-name-14f8d5f2-1175-4ba4-b72a-faf13eed4a46


Fold 5 C-index: 0.5727699530516432
[I 2024-04-13 23:59:38,537] Trial 0 finished with value: 0.6878264281814446 and parameters: {}. Best is trial 0 with value: 0.6878264281814446.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.6878264281814446], datetime_start=datetime.datetime(2024, 4, 13, 23, 59, 37, 313262), datetime_complete=datetime.datetime(2024, 4, 13, 23, 59, 38, 537482), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.6878264281814446


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.21397651688105565
Fold 2 IBS: 0.22157791055888598
Fold 3 IBS: 0.20453594148397225
Fold 4 IBS: 0.22473803361997696
Fold 5 IBS: 0.21812431365411103
[I 2024-04-13 23:59:40,856] Trial 0 finished with value: 0.2165905432396004 and parameters: {}. Best is trial 0 with value: 0.2165905432396004.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.2165905432396004], datetime_start=datetime.datetime(2024, 4, 13, 23, 59, 38, 972282), datetime_complete=datetime.datetime(2024, 4, 13, 23, 59, 40, 855896), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.2165905432396004


In [164]:
train_cindex['CoxRidge'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxRidge'] = np.round(study_ibs.best_value, 3)

In [165]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.688
train_ibs:  0.217


#### Test

In [166]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [167]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params['l1_ratio'] = 0.0000001
    best_params['fit_baseline_model']=True
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1e-07)

test_cindex : 0.534


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1e-07)

test_ibs:  0.221


In [168]:
# Saving the values to the dictionary 
test_cindex['CoxRidge'] = c_index
test_ibs['CoxRidge'] = ibs

### 3. CoxnetSurvivalAnalysis - Lasso

#### Train

In [169]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class(l1_ratio=1, 
                            fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Transformation
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            pt = PowerTransformer(method='yeo-johnson')
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = pt.fit_transform(X_train_included)
                X_test_included_std = pt.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-13 23:59:41,341] A new study created in memory with name: no-name-ea82e667-75a8-4bab-b21a-fef8e65b541e


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.7619047619047619
Fold 2 C-index: 0.7276785714285714
Fold 3 C-index: 0.8235294117647058
Fold 4 C-index: 0.8481012658227848


[I 2024-04-13 23:59:44,574] A new study created in memory with name: no-name-58580a70-0e85-4610-8e4c-716c0d11a753


Fold 5 C-index: 0.6384976525821596
[I 2024-04-13 23:59:44,556] Trial 0 finished with value: 0.7599423327005967 and parameters: {}. Best is trial 0 with value: 0.7599423327005967.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.7599423327005967], datetime_start=datetime.datetime(2024, 4, 13, 23, 59, 41, 849256), datetime_complete=datetime.datetime(2024, 4, 13, 23, 59, 44, 555255), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.7599423327005967


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.16110404192759542
Fold 2 IBS: 0.1850963138512818
Fold 3 IBS: 0.15349562060437788
Fold 4 IBS: 0.14737342192359726
Fold 5 IBS: 0.25731405171852934
[I 2024-04-13 23:59:49,033] Trial 0 finished with value: 0.1808766900050763 and parameters: {}. Best is trial 0 with value: 0.1808766900050763.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.1808766900050763], datetime_start=datetime.datetime(2024, 4, 13, 23, 59, 44, 764242), datetime_complete=datetime.datetime(2024, 4, 13, 23, 59, 49, 32499), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.1808766900050763


In [170]:
train_cindex['CoxLasso'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxLasso'] = np.round(study_ibs.best_value, 3)

In [171]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.76
train_ibs:  0.181


#### Test

In [172]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [173]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params['l1_ratio'] = 1
    best_params['fit_baseline_model']=True
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1)

test_cindex : 0.577


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1)

test_ibs:  0.269


In [174]:
# Saving the values to the dictionary 
test_cindex['CoxLasso'] = c_index
test_ibs['CoxLasso'] = ibs

### 4. CoxnetSurvivalAnalysis - ElasticNet

#### Train

In [175]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        l1_ratio = trial.suggest_float("l1_ratio", 0.0001, 1)
        
        # Create and fit survival model 
        model = model_class(l1_ratio=l1_ratio, 
                           fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Transformation
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            pt = PowerTransformer(method='yeo-johnson')
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = pt.fit_transform(X_train_included)
                X_test_included_std = pt.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-13 23:59:52,364] A new study created in memory with name: no-name-420bac82-1fae-400d-a748-193dad573a49


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.7619047619047619
Fold 2 C-index: 0.7321428571428571
Fold 3 C-index: 0.8235294117647058
Fold 4 C-index: 0.8481012658227848
Fold 5 C-index: 0.6338028169014085
[I 2024-04-13 23:59:55,997] Trial 0 finished with value: 0.7598962227073037 and parameters: {'l1_ratio': 0.6964995386793018}. Best is trial 0 with value: 0.7598962227073037.
Fold 1 C-index: 0.7619047619047619
Fold 2 C-index: 0.7321428571428571
Fold 3 C-index: 0.8235294117647058
Fold 4 C-index: 0.8481012658227848
Fold 5 C-index: 0.6338028169014085
[I 2024-04-13 23:59:58,891] Trial 1 finished with value: 0.7598962227073037 and parameters: {'l1_ratio': 0.28621072101688444}. Best is trial 0 with value: 0.7598962227073037.
Fold 1 C-index: 0.7619047619047619
Fold 2 C-index: 0.7321428571428571
Fold 3 C-index: 0.8235294117647058
Fold 4 C-index: 0.8481012658227848
Fold 5 C-index: 0.6338028169014085
[I 2024-04-14 00:00:03,509] Trial 2 finished with value: 0.7598962227073037 and parameters: {'l1_ratio': 0.22692876841884668}.

Fold 1 C-index: 0.7619047619047619
Fold 2 C-index: 0.7321428571428571
Fold 3 C-index: 0.8235294117647058
Fold 4 C-index: 0.8481012658227848
Fold 5 C-index: 0.6338028169014085
[I 2024-04-14 00:01:13,968] Trial 24 finished with value: 0.7598962227073037 and parameters: {'l1_ratio': 0.6311595421486845}. Best is trial 4 with value: 0.7608351898434539.
Fold 1 C-index: 0.7619047619047619
Fold 2 C-index: 0.7321428571428571
Fold 3 C-index: 0.8235294117647058
Fold 4 C-index: 0.8481012658227848
Fold 5 C-index: 0.6338028169014085
[I 2024-04-14 00:01:16,775] Trial 25 finished with value: 0.7598962227073037 and parameters: {'l1_ratio': 0.7477530840540471}. Best is trial 4 with value: 0.7608351898434539.
Fold 1 C-index: 0.7619047619047619
Fold 2 C-index: 0.7321428571428571
Fold 3 C-index: 0.8235294117647058
Fold 4 C-index: 0.8481012658227848
Fold 5 C-index: 0.6338028169014085
[I 2024-04-14 00:01:18,663] Trial 26 finished with value: 0.7598962227073037 and parameters: {'l1_ratio': 0.5655191009169166}

Fold 1 C-index: 0.7619047619047619
Fold 2 C-index: 0.7276785714285714
Fold 3 C-index: 0.8235294117647058
Fold 4 C-index: 0.8481012658227848
Fold 5 C-index: 0.6384976525821596
[I 2024-04-14 00:02:09,184] Trial 48 finished with value: 0.7599423327005967 and parameters: {'l1_ratio': 0.9631281987642258}. Best is trial 4 with value: 0.7608351898434539.
Fold 1 C-index: 0.7619047619047619
Fold 2 C-index: 0.7321428571428571
Fold 3 C-index: 0.8235294117647058
Fold 4 C-index: 0.8481012658227848
Fold 5 C-index: 0.6338028169014085
[I 2024-04-14 00:02:11,805] Trial 49 finished with value: 0.7598962227073037 and parameters: {'l1_ratio': 0.4477016529538363}. Best is trial 4 with value: 0.7608351898434539.
Fold 1 C-index: 0.7619047619047619
Fold 2 C-index: 0.7321428571428571
Fold 3 C-index: 0.8235294117647058
Fold 4 C-index: 0.8481012658227848
Fold 5 C-index: 0.6338028169014085
[I 2024-04-14 00:02:13,988] Trial 50 finished with value: 0.7598962227073037 and parameters: {'l1_ratio': 0.5938078108685914}

...

Fold 1 IBS: 0.16104778611325374
Fold 2 IBS: 0.18451024301927516
Fold 3 IBS: 0.15335484337078437
Fold 4 IBS: 0.1473613938635699
Fold 5 IBS: 0.25716919342656835
[I 2024-04-14 00:04:04,499] Trial 0 finished with value: 0.1806886919586903 and parameters: {'l1_ratio': 0.6964995386793018}. Best is trial 0 with value: 0.1806886919586903.
Fold 1 IBS: 0.1609248986949724
Fold 2 IBS: 0.18357347655213407
Fold 3 IBS: 0.15319254722217174
Fold 4 IBS: 0.14725671210972432
Fold 5 IBS: 0.25703159098488604
[I 2024-04-14 00:04:06,577] Trial 1 finished with value: 0.1803958451127777 and parameters: {'l1_ratio': 0.28621072101688444}. Best is trial 1 with value: 0.1803958451127777.
Fold 1 IBS: 0.1609083029281512
Fold 2 IBS: 0.18339581114281553
Fold 3 IBS: 0.1531353127892448
Fold 4 IBS: 0.1471872738245949
Fold 5 IBS: 0.2570843134945033
[I 2024-04-14 00:04:08,570] Trial 2 finished with value: 0.18034220283586194 and parameters: {'l1_ratio': 0.22692876841884668}. Best is trial 2 with value: 0.18034220283586194.


Fold 1 IBS: 0.16095633204662854
Fold 2 IBS: 0.18378571499753618
Fold 3 IBS: 0.15317979537938767
Fold 4 IBS: 0.14727206416174068
Fold 5 IBS: 0.25709241819825984
[I 2024-04-14 00:04:57,920] Trial 25 finished with value: 0.18045726495671058 and parameters: {'l1_ratio': 0.3804525958179419}. Best is trial 18 with value: 0.17979984651541625.
Fold 1 IBS: 0.1609447058146484
Fold 2 IBS: 0.18371527005793908
Fold 3 IBS: 0.15318722138516566
Fold 4 IBS: 0.1472042050958536
Fold 5 IBS: 0.25698679393803664
[I 2024-04-14 00:05:00,249] Trial 26 finished with value: 0.18040763925832864 and parameters: {'l1_ratio': 0.3312072877416866}. Best is trial 18 with value: 0.17979984651541625.
Fold 1 IBS: 0.16085788163181322
Fold 2 IBS: 0.18095123883742373
Fold 3 IBS: 0.15309395372044504
Fold 4 IBS: 0.14722766946238738
Fold 5 IBS: 0.2569221645669365
[I 2024-04-14 00:05:02,707] Trial 27 finished with value: 0.17981058164380118 and parameters: {'l1_ratio': 0.07046652721581821}. Best is trial 18 with value: 0.1797998

Fold 1 IBS: 0.16084399217043524
Fold 2 IBS: 0.18084266545097055
Fold 3 IBS: 0.1530380420028172
Fold 4 IBS: 0.2223318863687651
Fold 5 IBS: 0.2569678365850347
[I 2024-04-14 00:05:58,828] Trial 50 finished with value: 0.19480488451560457 and parameters: {'l1_ratio': 0.034744277134253675}. Best is trial 32 with value: 0.1797876201694899.
Fold 1 IBS: 0.16086029199227134
Fold 2 IBS: 0.1810193377508919
Fold 3 IBS: 0.1530902379382648
Fold 4 IBS: 0.14721559411026014
Fold 5 IBS: 0.2569079967367536
[I 2024-04-14 00:06:01,159] Trial 51 finished with value: 0.17981869170568837 and parameters: {'l1_ratio': 0.07664015002356084}. Best is trial 32 with value: 0.1797876201694899.
Fold 1 IBS: 0.21375089513713016
Fold 2 IBS: 0.2212829738184272
Fold 3 IBS: 0.20427598896848173
Fold 4 IBS: 0.22439248930226477
Fold 5 IBS: 0.21805335904331832
[I 2024-04-14 00:06:01,984] Trial 52 finished with value: 0.21635114125392443 and parameters: {'l1_ratio': 0.004509908798268153}. Best is trial 32 with value: 0.179787620

Fold 1 IBS: 0.1608555293405721
Fold 2 IBS: 0.18092429527294762
Fold 3 IBS: 0.15308563284934484
Fold 4 IBS: 0.14721648117820035
Fold 5 IBS: 0.25690449586435554
[I 2024-04-14 00:06:58,614] Trial 75 finished with value: 0.17979728690108412 and parameters: {'l1_ratio': 0.0637151684499458}. Best is trial 32 with value: 0.1797876201694899.
Fold 1 IBS: 0.16085354758420364
Fold 2 IBS: 0.180884481411374
Fold 3 IBS: 0.15308308294772466
Fold 4 IBS: 0.1472158208369685
Fold 5 IBS: 0.2569015546570554
[I 2024-04-14 00:07:01,189] Trial 76 finished with value: 0.17978769748746526 and parameters: {'l1_ratio': 0.05804597688293481}. Best is trial 32 with value: 0.1797876201694899.
Fold 1 IBS: 0.16085503615706354
Fold 2 IBS: 0.18079663162121018
Fold 3 IBS: 0.15304553499855655
Fold 4 IBS: 0.14726910837494794
Fold 5 IBS: 0.256969636149881
[I 2024-04-14 00:07:03,427] Trial 77 finished with value: 0.17978718946033184 and parameters: {'l1_ratio': 0.06039237081117726}. Best is trial 77 with value: 0.179787189460

Fold 5 IBS: 0.25702183130397105
[I 2024-04-14 00:07:51,503] Trial 99 finished with value: 0.17986729654329037 and parameters: {'l1_ratio': 0.10733829561035309}. Best is trial 77 with value: 0.17978718946033184.


* Best trial for IBS: 
 FrozenTrial(number=77, state=TrialState.COMPLETE, values=[0.17978718946033184], datetime_start=datetime.datetime(2024, 4, 14, 0, 7, 1, 207555), datetime_complete=datetime.datetime(2024, 4, 14, 0, 7, 3, 426492), params={'l1_ratio': 0.06039237081117726}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'l1_ratio': FloatDistribution(high=1.0, log=False, low=0.0001, step=None)}, trial_id=77, value=None)


* Best Score for IBS: 
 0.17978718946033184


In [176]:
train_cindex['CoxElastic'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxElastic'] = np.round(study_ibs.best_value, 3)

In [177]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.761
train_ibs:  0.18


#### Test

In [178]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [179]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    return model_class(**best_params, fit_baseline_model=True)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=0.7194970228885845)

test_cindex : 0.577


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=0.06039237081117726)

test_ibs:  0.269


In [180]:
# Saving the values to the dictionary 
test_cindex['CoxElastic'] = c_index
test_ibs['CoxElastic'] = ibs

### 5. Random Survival Forest

#### Train

In [213]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        oob_score = trial.suggest_categorical("oob_score", [True, False])
        max_samples = trial.suggest_float("max_samples", 0.1, 1.0) 
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None])
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        
        # Include warm_start for C-index optimization
        if metric == "c-index":
            warm_start = trial.suggest_categorical("warm_start", [True, False])
        else:
            warm_start = False  # Exclude warm_start for other metrics
        
        # Create and fit survival model 
        model = model_class(min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            max_leaf_nodes=max_leaf_nodes,
                            n_estimators=n_estimators, 
                            oob_score=oob_score,
                            warm_start=warm_start,
                            max_depth=max_depth,
                            max_features=max_features,
                            min_weight_fraction_leaf=min_weight_fraction_leaf, 
                            max_samples=max_samples, 
                            random_state=123)

        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])
            
            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(RandomSurvivalForest, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(RandomSurvivalForest, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-14 12:48:50,539] A new study created in memory with name: no-name-8879b552-aecf-45dd-9d37-ed2f6d8804ed


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.7575757575757576
Fold 2 C-index: 0.609375
Fold 3 C-index: 0.7916666666666666
Fold 4 C-index: 0.8459915611814346
Fold 5 C-index: 0.636150234741784
[I 2024-04-14 12:48:53,950] Trial 0 finished with value: 0.7281518440331286 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 7, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 360, 'oob_score': False, 'max_samples': 0.7163467647263769, 'max_features': None, 'min_weight_fraction_leaf': 0.2192861223398122, 'warm_start': False}. Best is trial 0 with value: 0.7281518440331286.
Fold 1 C-index: 0.7683982683982684
Fold 2 C-index: 0.6517857142857143
Fold 3 C-index: 0.8088235294117647
Fold 4 C-index: 0.8565400843881856
Fold 5 C-index: 0.6431924882629108
[I 2024-04-14 12:48:56,941] Trial 1 finished with value: 0.7457480169493687 and parameters: {'min_samples_split': 16, 'max_leaf_nodes': 5, 'min_samples_leaf': 4, 'max_depth': 11, 'n_estimators': 266, 'oob_score': False, 'max_samples': 0.7520097923745717, 'max_featur

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-14 12:49:37,255] Trial 16 finished with value: 0.5 and parameters: {'min_samples_split': 7, 'max_leaf_nodes': 17, 'min_samples_leaf': 10, 'max_depth': 1, 'n_estimators': 122, 'oob_score': True, 'max_samples': 0.5327960974085133, 'max_features': 'sqrt', 'min_weight_fraction_leaf': 0.38557060567081813, 'warm_start': True}. Best is trial 14 with value: 0.7600266365278019.
Fold 1 C-index: 0.79004329004329
Fold 2 C-index: 0.7566964285714286
Fold 3 C-index: 0.8480392156862745
Fold 4 C-index: 0.8565400843881856
Fold 5 C-index: 0.6948356807511737
[I 2024-04-14 12:49:37,477] Trial 17 finished with value: 0.7892309398880706 and parameters: {'min_samples_split': 12, 'max_leaf_nodes': 8, 'min_samples_leaf': 7, 'max_depth': 7, 'n_estimators': 10, 'oob_score': True, 'max_samples': 0.4160888955210671, 'max_features': 'sqrt', 'min_weight_fraction_leaf': 0.06727307964156963, 'warm_start': True

Fold 1 C-index: 0.7380952380952381
Fold 2 C-index: 0.7857142857142857
Fold 3 C-index: 0.8382352941176471
Fold 4 C-index: 0.8649789029535865
Fold 5 C-index: 0.6948356807511737
[I 2024-04-14 12:49:46,830] Trial 32 finished with value: 0.7843718803263862 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 6, 'min_samples_leaf': 9, 'max_depth': 16, 'n_estimators': 77, 'oob_score': True, 'max_samples': 0.5438109085992663, 'max_features': None, 'min_weight_fraction_leaf': 0.08648554774122166, 'warm_start': True}. Best is trial 17 with value: 0.7892309398880706.
Fold 1 C-index: 0.7554112554112554
Fold 2 C-index: 0.7723214285714286
Fold 3 C-index: 0.8088235294117647
Fold 4 C-index: 0.869198312236287
Fold 5 C-index: 0.6643192488262911
[I 2024-04-14 12:49:47,932] Trial 33 finished with value: 0.7740147548914054 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 4, 'min_samples_leaf': 8, 'max_depth': 17, 'n_estimators': 156, 'oob_score': True, 'max_samples': 0.544101480226597, 'max

...

Fold 1 IBS: 0.21398563901141657
Fold 2 IBS: 0.22134596740550985
Fold 3 IBS: 0.2048025990579434
Fold 4 IBS: 0.22461415055237854
Fold 5 IBS: 0.21851803782809134
[I 2024-04-14 12:53:39,400] Trial 32 finished with value: 0.21665327877106794 and parameters: {'min_samples_split': 13, 'max_leaf_nodes': 10, 'min_samples_leaf': 3, 'max_depth': 18, 'n_estimators': 451, 'oob_score': False, 'max_samples': 0.7396444271446992, 'max_features': None, 'min_weight_fraction_leaf': 0.4501495537267681}. Best is trial 23 with value: 0.18256840611756237.
Fold 1 IBS: 0.1881712152839018
Fold 2 IBS: 0.19602908273271186
Fold 3 IBS: 0.187050921824735
Fold 4 IBS: 0.18923643256610323
Fold 5 IBS: 0.21856358775020004
[I 2024-04-14 12:53:43,871] Trial 33 finished with value: 0.19581024803153038 and parameters: {'min_samples_split': 7, 'max_leaf_nodes': 12, 'min_samples_leaf': 1, 'max_depth': 19, 'n_estimators': 493, 'oob_score': False, 'max_samples': 0.9933327757837453, 'max_features': None, 'min_weight_fraction_leaf'

Fold 1 IBS: 0.1705674173139149
Fold 2 IBS: 0.1852617720416809
Fold 3 IBS: 0.16929920721077502
Fold 4 IBS: 0.16857228614310335
Fold 5 IBS: 0.23285081643644664
[I 2024-04-14 12:54:41,330] Trial 48 finished with value: 0.18531029982918418 and parameters: {'min_samples_split': 9, 'max_leaf_nodes': 13, 'min_samples_leaf': 5, 'max_depth': 15, 'n_estimators': 431, 'oob_score': False, 'max_samples': 0.9996528707471266, 'max_features': None, 'min_weight_fraction_leaf': 0.47380170342998185}. Best is trial 23 with value: 0.18256840611756237.
Fold 1 IBS: 0.21397420828984268
Fold 2 IBS: 0.22140730914697682
Fold 3 IBS: 0.2048165899556378
Fold 4 IBS: 0.22466454285026752
Fold 5 IBS: 0.2185528446348627
[I 2024-04-14 12:54:44,248] Trial 49 finished with value: 0.21668309897551746 and parameters: {'min_samples_split': 12, 'max_leaf_nodes': 8, 'min_samples_leaf': 20, 'max_depth': 6, 'n_estimators': 330, 'oob_score': False, 'max_samples': 0.8459565404966409, 'max_features': 'auto', 'min_weight_fraction_lea

Fold 1 IBS: 0.17758210577267103
Fold 2 IBS: 0.18839515678161117
Fold 3 IBS: 0.17378027525748194
Fold 4 IBS: 0.17402263926693903
Fold 5 IBS: 0.23329928868140723
[I 2024-04-14 12:55:41,730] Trial 64 finished with value: 0.1894158931520221 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 10, 'min_samples_leaf': 3, 'max_depth': 20, 'n_estimators': 421, 'oob_score': False, 'max_samples': 0.8662053800926933, 'max_features': None, 'min_weight_fraction_leaf': 0.4182804957604697}. Best is trial 54 with value: 0.1822436647018153.
Fold 1 IBS: 0.17735779335885185
Fold 2 IBS: 0.18791831745948914
Fold 3 IBS: 0.17437629608751362
Fold 4 IBS: 0.1754060616546171
Fold 5 IBS: 0.2347254760738243
[I 2024-04-14 12:55:45,039] Trial 65 finished with value: 0.18995678892685922 and parameters: {'min_samples_split': 4, 'max_leaf_nodes': 12, 'min_samples_leaf': 7, 'max_depth': 18, 'n_estimators': 380, 'oob_score': False, 'max_samples': 0.9168825987611354, 'max_features': None, 'min_weight_fraction_leaf':

Fold 1 IBS: 0.1754183266894564
Fold 2 IBS: 0.2119163985327653
Fold 3 IBS: 0.16611082563106694
Fold 4 IBS: 0.16255778819587066
Fold 5 IBS: 0.22916546086072462
[I 2024-04-14 12:56:40,234] Trial 80 finished with value: 0.18903375998197677 and parameters: {'min_samples_split': 8, 'max_leaf_nodes': 8, 'min_samples_leaf': 3, 'max_depth': 12, 'n_estimators': 237, 'oob_score': False, 'max_samples': 0.9992480396864105, 'max_features': None, 'min_weight_fraction_leaf': 0.40510470687091726}. Best is trial 54 with value: 0.1822436647018153.
Fold 1 IBS: 0.16993730570122387
Fold 2 IBS: 0.195387539069006
Fold 3 IBS: 0.1651714087990093
Fold 4 IBS: 0.16201193892631496
Fold 5 IBS: 0.22567140464684687
[I 2024-04-14 12:56:43,067] Trial 81 finished with value: 0.18363591942848018 and parameters: {'min_samples_split': 2, 'max_leaf_nodes': 16, 'min_samples_leaf': 7, 'max_depth': 15, 'n_estimators': 296, 'oob_score': False, 'max_samples': 0.8729504953795615, 'max_features': None, 'min_weight_fraction_leaf': 0

Fold 1 IBS: 0.17691142196188117
Fold 2 IBS: 0.1884356638717461
Fold 3 IBS: 0.17473493092732248
Fold 4 IBS: 0.1743581409278057
Fold 5 IBS: 0.23455157057349743
[I 2024-04-14 12:57:35,826] Trial 96 finished with value: 0.18979834565245057 and parameters: {'min_samples_split': 7, 'max_leaf_nodes': 15, 'min_samples_leaf': 6, 'max_depth': 2, 'n_estimators': 412, 'oob_score': False, 'max_samples': 0.9548184208743441, 'max_features': None, 'min_weight_fraction_leaf': 0.46352447817689174}. Best is trial 94 with value: 0.18218245586616172.
Fold 1 IBS: 0.1720334166380091
Fold 2 IBS: 0.19310219127184633
Fold 3 IBS: 0.16467180888297991
Fold 4 IBS: 0.16215325103272105
Fold 5 IBS: 0.225731258677061
[I 2024-04-14 12:57:39,841] Trial 97 finished with value: 0.1835383853005235 and parameters: {'min_samples_split': 6, 'max_leaf_nodes': 14, 'min_samples_leaf': 8, 'max_depth': 2, 'n_estimators': 434, 'oob_score': False, 'max_samples': 0.9862595341313025, 'max_features': None, 'min_weight_fraction_leaf': 0.

In [214]:
train_cindex['Randomsurvivalforest'] = np.round(study_cindex.best_value, 3)
train_ibs['Randomsurvivalforest'] = np.round(study_ibs.best_value, 3)

In [215]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.855
train_ibs:  0.182


#### Test

In [216]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))
    
y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [217]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(RandomSurvivalForest, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex: ", c_index)

# Set the best model 
best_model_ibs = create_best_model(RandomSurvivalForest, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

RandomSurvivalForest(max_depth=14, max_features=None, max_leaf_nodes=11,
                     max_samples=0.7127178148051858, min_samples_leaf=1,
                     min_samples_split=8,
                     min_weight_fraction_leaf=0.01586498778855871,
                     n_estimators=236, random_state=123, warm_start=True)

test_cindex:  0.643


RandomSurvivalForest(max_depth=2, max_features=None, max_leaf_nodes=13,
                     max_samples=0.9526180318811873, min_samples_leaf=6,
                     min_samples_split=7,
                     min_weight_fraction_leaf=0.4336258803047899,
                     n_estimators=408, random_state=123)

test_ibs:  0.206


In [218]:
# Saving the values to the dictionary 
test_cindex['Randomsurvivalforest'] = c_index
test_ibs['Randomsurvivalforest'] = ibs

### 6. ExtraSurvivalTrees

#### Train

In [219]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

In [220]:
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters 
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        oob_score = trial.suggest_categorical("oob_score", [True, False])
        warm_start = trial.suggest_categorical("warm_start", [True, False])
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None, 0.1, 1])
        max_samples = trial.suggest_float("max_samples", 0.1, 1.0) 
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        
        # Include warm_start for C-index optimization
        if metric == "c-index":
            warm_start = trial.suggest_categorical("warm_start", [True, False])
        else:
            warm_start = False  # Exclude warm_start for other metrics

        
        # Create and fit survival model 
        model = model_class(min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            max_leaf_nodes=max_leaf_nodes,
                            n_estimators=n_estimators, 
                            oob_score=oob_score, 
                            max_features=max_features, 
                            warm_start=warm_start, 
                            max_samples=max_samples,
                            min_weight_fraction_leaf=min_weight_fraction_leaf, 
                            max_depth=max_depth, 
                            random_state=123) 
        
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(ExtraSurvivalTrees, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(ExtraSurvivalTrees, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-14 12:57:48,645] A new study created in memory with name: no-name-56a079ec-81c4-4472-b2db-d648da77fd08


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.7640692640692641
Fold 2 C-index: 0.7589285714285714
Fold 3 C-index: 0.8431372549019608
Fold 4 C-index: 0.8354430379746836
Fold 5 C-index: 0.6713615023474179
[I 2024-04-14 12:57:49,605] Trial 0 finished with value: 0.7745879261443795 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 7, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 360, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.7641958651588321, 'min_weight_fraction_leaf': 0.09124586522674999}. Best is trial 0 with value: 0.7745879261443795.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-14 12:57:51,614] Trial 1 finished with value: 0.5 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 12, 'min_samples_leaf': 11, 'max_depth': 13, 'n_estimators': 425, 'oob_score': True, 'warm_start': True, 'max_features': None, 'max_samples': 0.4877764869966794, 'min_weight_fraction_leaf': 0.2468425488251531

Fold 1 C-index: 0.7467532467532467
Fold 2 C-index: 0.8035714285714286
Fold 3 C-index: 0.8529411764705882
Fold 4 C-index: 0.8565400843881856
Fold 5 C-index: 0.6549295774647887
[I 2024-04-14 12:58:13,247] Trial 16 finished with value: 0.7829471027296476 and parameters: {'min_samples_split': 12, 'max_leaf_nodes': 10, 'min_samples_leaf': 7, 'max_depth': 20, 'n_estimators': 237, 'oob_score': False, 'warm_start': True, 'max_features': None, 'max_samples': 0.8988170687511211, 'min_weight_fraction_leaf': 0.1144909881867055}. Best is trial 16 with value: 0.7829471027296476.
Fold 1 C-index: 0.7662337662337663
Fold 2 C-index: 0.8035714285714286
Fold 3 C-index: 0.8627450980392157
Fold 4 C-index: 0.8523206751054853
Fold 5 C-index: 0.6666666666666666
[I 2024-04-14 12:58:14,005] Trial 17 finished with value: 0.7903075269233125 and parameters: {'min_samples_split': 8, 'max_leaf_nodes': 10, 'min_samples_leaf': 6, 'max_depth': 17, 'n_estimators': 256, 'oob_score': False, 'warm_start': True, 'max_feature

Fold 1 C-index: 0.7532467532467533
Fold 2 C-index: 0.8125
Fold 3 C-index: 0.8676470588235294
Fold 4 C-index: 0.8438818565400844
Fold 5 C-index: 0.6713615023474179
[I 2024-04-14 12:58:26,200] Trial 31 finished with value: 0.789727434191557 and parameters: {'min_samples_split': 8, 'max_leaf_nodes': 8, 'min_samples_leaf': 4, 'max_depth': 17, 'n_estimators': 235, 'oob_score': False, 'warm_start': True, 'max_features': None, 'max_samples': 0.9117101001226512, 'min_weight_fraction_leaf': 0.07589392630375186}. Best is trial 25 with value: 0.8076326900231233.
Fold 1 C-index: 0.7575757575757576
Fold 2 C-index: 0.8214285714285714
Fold 3 C-index: 0.8774509803921569
Fold 4 C-index: 0.8481012658227848
Fold 5 C-index: 0.6948356807511737
[I 2024-04-14 12:58:26,972] Trial 32 finished with value: 0.7998784511940888 and parameters: {'min_samples_split': 7, 'max_leaf_nodes': 9, 'min_samples_leaf': 4, 'max_depth': 13, 'n_estimators': 236, 'oob_score': False, 'warm_start': True, 'max_features': None, 'max_

...

Fold 2 IBS: 0.2145204309047712
Fold 3 IBS: 0.16344681522379592
Fold 4 IBS: 0.1607591766490642
Fold 5 IBS: 0.23285217315239765
[I 2024-04-14 13:00:26,822] Trial 30 finished with value: 0.18689210617143953 and parameters: {'min_samples_split': 11, 'max_leaf_nodes': 19, 'min_samples_leaf': 4, 'max_depth': 7, 'n_estimators': 87, 'oob_score': False, 'warm_start': False, 'max_features': 'sqrt', 'max_samples': 0.9504933061433887, 'min_weight_fraction_leaf': 0.1352883886134873}. Best is trial 7 with value: 0.18575431268869705.
Fold 1 IBS: 0.1522561822367273
Fold 2 IBS: 0.22165813527341477
Fold 3 IBS: 0.15806345503003275
Fold 4 IBS: 0.14643989125013568
Fold 5 IBS: 0.23525263253654355
[I 2024-04-14 13:00:28,347] Trial 31 finished with value: 0.18273405926537079 and parameters: {'min_samples_split': 12, 'max_leaf_nodes': 16, 'min_samples_leaf': 2, 'max_depth': 5, 'n_estimators': 197, 'oob_score': False, 'warm_start': False, 'max_features': 'log2', 'max_samples': 0.9254246685599902, 'min_weight_fr

Fold 1 IBS: 0.154877427493089
Fold 2 IBS: 0.2119381972371029
Fold 3 IBS: 0.16230571380133155
Fold 4 IBS: 0.15585982768129883
Fold 5 IBS: 0.23020111518690609
[I 2024-04-14 13:00:50,061] Trial 45 finished with value: 0.18303645627994566 and parameters: {'min_samples_split': 10, 'max_leaf_nodes': 7, 'min_samples_leaf': 2, 'max_depth': 5, 'n_estimators': 139, 'oob_score': False, 'warm_start': False, 'max_features': 0.1, 'max_samples': 0.7784599232837964, 'min_weight_fraction_leaf': 0.0038388411371728937}. Best is trial 31 with value: 0.18273405926537079.
Fold 1 IBS: 0.1807341100446867
Fold 2 IBS: 0.20797545001135234
Fold 3 IBS: 0.17776524098244717
Fold 4 IBS: 0.18082034578648548
Fold 5 IBS: 0.22062001984045795
[I 2024-04-14 13:00:51,053] Trial 46 finished with value: 0.19358303333308596 and parameters: {'min_samples_split': 10, 'max_leaf_nodes': 7, 'min_samples_leaf': 12, 'max_depth': 3, 'n_estimators': 138, 'oob_score': False, 'warm_start': False, 'max_features': 0.1, 'max_samples': 0.783

Fold 1 IBS: 0.1667377112785077
Fold 2 IBS: 0.21139971336963634
Fold 3 IBS: 0.16719610422116873
Fold 4 IBS: 0.16444742789928804
Fold 5 IBS: 0.22433929414894066
[I 2024-04-14 13:01:13,416] Trial 60 finished with value: 0.18682405018350828 and parameters: {'min_samples_split': 14, 'max_leaf_nodes': 7, 'min_samples_leaf': 6, 'max_depth': 5, 'n_estimators': 233, 'oob_score': False, 'warm_start': False, 'max_features': 0.1, 'max_samples': 0.9208992051748564, 'min_weight_fraction_leaf': 0.04194475489343627}. Best is trial 31 with value: 0.18273405926537079.
Fold 1 IBS: 0.15339272842252286
Fold 2 IBS: 0.22875929926363106
Fold 3 IBS: 0.15824814328892167
Fold 4 IBS: 0.14551047397439773
Fold 5 IBS: 0.23751514003516505
[I 2024-04-14 13:01:14,769] Trial 61 finished with value: 0.1846851569969277 and parameters: {'min_samples_split': 7, 'max_leaf_nodes': 16, 'min_samples_leaf': 2, 'max_depth': 9, 'n_estimators': 182, 'oob_score': False, 'warm_start': False, 'max_features': 'log2', 'max_samples': 0.8

Fold 1 IBS: 0.15528894958073775
Fold 2 IBS: 0.21984680247745453
Fold 3 IBS: 0.15900685739714732
Fold 4 IBS: 0.14859781459794244
Fold 5 IBS: 0.23493449886479167
[I 2024-04-14 13:01:38,731] Trial 75 finished with value: 0.18353498458361475 and parameters: {'min_samples_split': 16, 'max_leaf_nodes': 16, 'min_samples_leaf': 3, 'max_depth': 5, 'n_estimators': 162, 'oob_score': True, 'warm_start': False, 'max_features': 'log2', 'max_samples': 0.9532062735167633, 'min_weight_fraction_leaf': 0.030316433196080697}. Best is trial 31 with value: 0.18273405926537079.
Fold 1 IBS: 0.1635808951920651
Fold 2 IBS: 0.20759879371938625
Fold 3 IBS: 0.16635458220500324
Fold 4 IBS: 0.1603635273963648
Fold 5 IBS: 0.2261410117149779
[I 2024-04-14 13:01:40,000] Trial 76 finished with value: 0.18480776204555946 and parameters: {'min_samples_split': 17, 'max_leaf_nodes': 9, 'min_samples_leaf': 4, 'max_depth': 5, 'n_estimators': 128, 'oob_score': True, 'warm_start': False, 'max_features': 0.1, 'max_samples': 0.96

Fold 2 IBS: 0.2063565222747697
Fold 3 IBS: 0.17003588715814522
Fold 4 IBS: 0.17777116278712887
Fold 5 IBS: 0.22022528125047466
[I 2024-04-14 13:01:49,962] Trial 90 finished with value: 0.18995479834281107 and parameters: {'min_samples_split': 19, 'max_leaf_nodes': 17, 'min_samples_leaf': 1, 'max_depth': 4, 'n_estimators': 37, 'oob_score': True, 'warm_start': True, 'max_features': 'sqrt', 'max_samples': 0.9997596356740963, 'min_weight_fraction_leaf': 0.3292134206382089}. Best is trial 81 with value: 0.1804545979151259.
Fold 1 IBS: 0.15458861367684845
Fold 2 IBS: 0.21750786042177303
Fold 3 IBS: 0.15767300132028467
Fold 4 IBS: 0.15084191777051614
Fold 5 IBS: 0.23638504482176886
[I 2024-04-14 13:01:50,617] Trial 91 finished with value: 0.18339928760223825 and parameters: {'min_samples_split': 17, 'max_leaf_nodes': 18, 'min_samples_leaf': 2, 'max_depth': 6, 'n_estimators': 60, 'oob_score': True, 'warm_start': False, 'max_features': 'sqrt', 'max_samples': 0.916420719730876, 'min_weight_fract

In [221]:
train_cindex['ExtraSurvivalTrees'] = np.round(study_cindex.best_value, 3)
train_ibs['ExtraSurvivalTrees'] = np.round(study_ibs.best_value, 3)

In [222]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.848
train_ibs:  0.18


#### Test

In [223]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [224]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(ExtraSurvivalTrees, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(ExtraSurvivalTrees, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

ExtraSurvivalTrees(max_depth=10, max_features=None, max_leaf_nodes=17,
                   max_samples=0.7657259832611387, min_samples_leaf=1,
                   min_samples_split=2,
                   min_weight_fraction_leaf=0.007259167100140962,
                   n_estimators=358, random_state=123, warm_start=True)

C-index score: 0.61


ExtraSurvivalTrees(max_depth=5, max_leaf_nodes=17,
                   max_samples=0.9277129016361768, min_samples_leaf=1,
                   min_samples_split=16,
                   min_weight_fraction_leaf=0.026154087319717897,
                   n_estimators=27, oob_score=True, random_state=123)

IBS: 0.23


In [225]:
# Saving the values to the dictionary 
test_cindex['ExtraSurvivalTrees'] = c_index
test_ibs['ExtraSurvivalTrees'] = ibs

### 7. GradientBoostingSurvivalAnalysis

#### Train

In [226]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

# Running to optuna for hyperparameter tuning
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        subsample = trial.suggest_float("subsample", 0.1, 1)
        learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
        dropout_rate = trial.suggest_float("dropout_rate", 0.1, 1)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        criterion = trial.suggest_categorical('criterion', ['friedman_mse', 'squared_error'])
        ccp_alpha = trial.suggest_float("ccp_alpha", 0.0, 10)
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None, 0.1, 1])
        min_impurity_decrease = trial.suggest_loguniform('min_impurity_decrease', 1e-7, 1e-1)
        validation_fraction = trial.suggest_float("validation_fraction", 0.0, 1.0)
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        
        # Create and fit survival model 
        model = model_class(subsample=subsample,
                            learning_rate=learning_rate,
                            dropout_rate=dropout_rate,
                            n_estimators=n_estimators,
                            ccp_alpha=ccp_alpha, 
                            criterion=criterion,
                            min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            min_weight_fraction_leaf=min_weight_fraction_leaf,
                            max_depth=max_depth,
                            max_features=max_features,
                            max_leaf_nodes=max_leaf_nodes, 
                            min_impurity_decrease=min_impurity_decrease,
                            validation_fraction=validation_fraction, 
                            random_state=123)
        
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(GradientBoostingSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(GradientBoostingSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-14 13:01:55,457] A new study created in memory with name: no-name-06736fcc-37e4-4f39-acb6-6f63d46f2273


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-14 13:02:09,208] Trial 0 finished with value: 0.5 and parameters: {'subsample': 0.7268222670380755, 'learning_rate': 0.02932779416008757, 'dropout_rate': 0.3041663082077828, 'n_estimators': 276, 'criterion': 'friedman_mse', 'ccp_alpha': 9.807641983846155, 'min_weight_fraction_leaf': 0.34241486929243165, 'max_features': None, 'min_impurity_decrease': 2.4449249473284515e-05, 'validation_fraction': 0.7379954057320357, 'min_samples_split': 5, 'max_leaf_nodes': 5, 'min_samples_leaf': 11, 'max_depth': 11}. Best is trial 0 with value: 0.5.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-14 13:02:16,625] Trial 1 finished with value: 0.5 and parameters: {'subsample': 0.6709608626961889, 'learning_rate': 0.08509374761370117, 'dropout_rate': 0.7520097923745717, 'n_estimators': 306, 'criterion': 'friedman_mse', 'ccp_alpha': 3.

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-14 13:47:23,712] Trial 13 finished with value: 0.5 and parameters: {'subsample': 0.8386796524426539, 'learning_rate': 0.046734492485875676, 'dropout_rate': 0.4821375662037144, 'n_estimators': 402, 'criterion': 'squared_error', 'ccp_alpha': 1.5696007313501796, 'min_weight_fraction_leaf': 0.18684147934268416, 'max_features': 'log2', 'min_impurity_decrease': 1.5044881127471587e-06, 'validation_fraction': 0.8166356053932342, 'min_samples_split': 16, 'max_leaf_nodes': 19, 'min_samples_leaf': 16, 'max_depth': 4}. Best is trial 9 with value: 0.7308620854904466.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-14 14:28:28,155] Trial 14 finished with value: 0.5 and parameters: {'subsample': 0.33235389014851724, 'learning_rate': 0.04522573411670834, 'dropout_rate': 0.2712811374536856, 'n_estimators': 405, 'criterion': 'square

Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-14 14:39:16,214] Trial 25 finished with value: 0.5 and parameters: {'subsample': 0.7565257046780437, 'learning_rate': 0.01188344684416992, 'dropout_rate': 0.3433523077110169, 'n_estimators': 318, 'criterion': 'squared_error', 'ccp_alpha': 2.063559212730723, 'min_weight_fraction_leaf': 0.25401273903421573, 'max_features': 'auto', 'min_impurity_decrease': 5.773435946665558e-07, 'validation_fraction': 0.8062268184400477, 'min_samples_split': 16, 'max_leaf_nodes': 18, 'min_samples_leaf': 5, 'max_depth': 3}. Best is trial 9 with value: 0.7308620854904466.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.6004464285714286
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.6197183098591549
[I 2024-04-14 14:39:53,321] Trial 26 finished with value: 0.5440329476861168 and parameters: {'subsample': 0.8922404482621683, 'learning_rate': 0.011585260292674536, 'dropout_rate': 0.2527000999648632, 'n_estimators': 446,

...

Fold 2 IBS: 0.22157791718667935
Fold 3 IBS: 0.20453594732018135
Fold 4 IBS: 0.22473804139762676
Fold 5 IBS: 0.21812431525609566
[I 2024-04-14 15:26:06,954] Trial 55 finished with value: 0.21659054862241583 and parameters: {'subsample': 0.9027683411994925, 'learning_rate': 0.023646082998228058, 'dropout_rate': 0.13472529918097312, 'n_estimators': 381, 'criterion': 'squared_error', 'ccp_alpha': 1.3381569877935875, 'min_weight_fraction_leaf': 0.36938177041618503, 'max_features': 'auto', 'min_impurity_decrease': 1.1016843774656315e-07, 'validation_fraction': 0.898549324711475, 'min_samples_split': 3, 'max_leaf_nodes': 12, 'min_samples_leaf': 12, 'max_depth': 3}. Best is trial 53 with value: 0.21504716009286912.
Fold 1 IBS: 0.21397652195149616
Fold 2 IBS: 0.22157791718667938
Fold 3 IBS: 0.20453594732018135
Fold 4 IBS: 0.2247380413976268
Fold 5 IBS: 0.21812431525609582
[I 2024-04-14 15:26:38,992] Trial 56 finished with value: 0.21659054862241592 and parameters: {'subsample': 0.83247895380525

Fold 2 IBS: 0.22157791718667938
Fold 3 IBS: 0.20453594732018132
Fold 4 IBS: 0.2247380413976268
Fold 5 IBS: 0.2181243152560957
[I 2024-04-14 15:34:42,052] Trial 66 finished with value: 0.21659054862241586 and parameters: {'subsample': 0.9998769156045215, 'learning_rate': 0.008782667792366316, 'dropout_rate': 0.12972298735589705, 'n_estimators': 500, 'criterion': 'squared_error', 'ccp_alpha': 0.7134499429690735, 'min_weight_fraction_leaf': 0.19311178079767077, 'max_features': 'log2', 'min_impurity_decrease': 1.142816958470248e-06, 'validation_fraction': 0.9729459657934212, 'min_samples_split': 18, 'max_leaf_nodes': 16, 'min_samples_leaf': 8, 'max_depth': 4}. Best is trial 53 with value: 0.21504716009286912.
Fold 1 IBS: 0.21397652195149613
Fold 2 IBS: 0.22157791718667935
Fold 3 IBS: 0.20453594732018138
Fold 4 IBS: 0.22473804139762676
Fold 5 IBS: 0.21812431525609582
[I 2024-04-14 15:35:28,216] Trial 67 finished with value: 0.21659054862241592 and parameters: {'subsample': 0.864018533057457

Fold 1 IBS: 0.21397652195149613
Fold 2 IBS: 0.22157791718667938
Fold 3 IBS: 0.20453594732018135
Fold 4 IBS: 0.2247380413976268
Fold 5 IBS: 0.2181243152560957
[I 2024-04-14 15:42:17,047] Trial 77 finished with value: 0.21659054862241586 and parameters: {'subsample': 0.6527746876723795, 'learning_rate': 0.05776064499915696, 'dropout_rate': 0.2556920821676466, 'n_estimators': 442, 'criterion': 'squared_error', 'ccp_alpha': 0.5896250392603071, 'min_weight_fraction_leaf': 0.22988050749684658, 'max_features': 1, 'min_impurity_decrease': 1.4995945809430595e-05, 'validation_fraction': 0.8950354974332698, 'min_samples_split': 19, 'max_leaf_nodes': 16, 'min_samples_leaf': 11, 'max_depth': 3}. Best is trial 53 with value: 0.21504716009286912.
Fold 1 IBS: 0.21397652195149613
Fold 2 IBS: 0.22157791718667938
Fold 3 IBS: 0.20453594732018138
Fold 4 IBS: 0.22473804139762676
Fold 5 IBS: 0.21812431525609566
[I 2024-04-14 15:43:04,632] Trial 78 finished with value: 0.21659054862241586 and parameters: {'su

Fold 1 IBS: 0.21397652195149616
Fold 2 IBS: 0.22157791718667938
Fold 3 IBS: 0.20453594732018135
Fold 4 IBS: 0.2247380413976268
Fold 5 IBS: 0.2181243152560957
[I 2024-04-14 15:50:08,914] Trial 88 finished with value: 0.21659054862241586 and parameters: {'subsample': 0.8800231052433968, 'learning_rate': 0.015424283294721588, 'dropout_rate': 0.20093805843087115, 'n_estimators': 390, 'criterion': 'squared_error', 'ccp_alpha': 1.351539840282731, 'min_weight_fraction_leaf': 0.28958635214901884, 'max_features': 'auto', 'min_impurity_decrease': 2.329078771156988e-07, 'validation_fraction': 0.939344740485312, 'min_samples_split': 6, 'max_leaf_nodes': 15, 'min_samples_leaf': 7, 'max_depth': 4}. Best is trial 53 with value: 0.21504716009286912.
Fold 1 IBS: 0.21397652195149616
Fold 2 IBS: 0.22157791718667935
Fold 3 IBS: 0.20453594732018138
Fold 4 IBS: 0.22473804139762676
Fold 5 IBS: 0.21812431525609566
[I 2024-04-14 15:50:43,436] Trial 89 finished with value: 0.21659054862241586 and parameters: {'

Fold 1 IBS: 0.21252354672208404
Fold 2 IBS: 0.22052727078684145
Fold 3 IBS: 0.20320794231873263
Fold 4 IBS: 0.2228335466693306
Fold 5 IBS: 0.21782914318597227
[I 2024-04-14 15:58:40,038] Trial 99 finished with value: 0.21538428993659223 and parameters: {'subsample': 0.24286963781937687, 'learning_rate': 0.01358129947690588, 'dropout_rate': 0.25496540816023416, 'n_estimators': 454, 'criterion': 'squared_error', 'ccp_alpha': 0.004367160733265784, 'min_weight_fraction_leaf': 0.2624099419664444, 'max_features': 'auto', 'min_impurity_decrease': 5.321953542697131e-07, 'validation_fraction': 0.9620562032012763, 'min_samples_split': 11, 'max_leaf_nodes': 20, 'min_samples_leaf': 10, 'max_depth': 1}. Best is trial 53 with value: 0.21504716009286912.


* Best trial for IBS: 
 FrozenTrial(number=53, state=TrialState.COMPLETE, values=[0.21504716009286912], datetime_start=datetime.datetime(2024, 4, 14, 15, 23, 58, 526416), datetime_complete=datetime.datetime(2024, 4, 14, 15, 24, 41, 86112), params={

In [227]:
train_cindex['GradientBoosting'] = np.round(study_cindex.best_value, 3)
train_ibs['GradientBoosting'] = np.round(study_ibs.best_value, 3)

In [228]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.74
train_ibs:  0.215


#### Test

In [229]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [230]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(GradientBoostingSurvivalAnalysis, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(GradientBoostingSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

GradientBoostingSurvivalAnalysis(ccp_alpha=0.03063070291051248,
                                 criterion='squared_error',
                                 dropout_rate=0.2576884847115747,
                                 learning_rate=0.007359366951045268,
                                 max_depth=16, max_features=1,
                                 max_leaf_nodes=16,
                                 min_impurity_decrease=1.3903906488794697e-07,
                                 min_samples_leaf=14, min_samples_split=20,
                                 min_weight_fraction_leaf=0.38180626899357545,
                                 n_estimators=96, random_state=123,
                                 subsample=0.8938290428827321,
                                 validation_fraction=0.9577535215137098)

C-index score: 0.612


GradientBoostingSurvivalAnalysis(ccp_alpha=0.009625013743012712,
                                 criterion='squared_error',
                                 dropout_rate=0.1897783294507234,
                                 learning_rate=0.015420772490455037,
                                 max_features='auto', max_leaf_nodes=14,
                                 min_impurity_decrease=5.869825897765074e-07,
                                 min_samples_leaf=13, min_samples_split=20,
                                 min_weight_fraction_leaf=0.29880213170319914,
                                 n_estimators=430, random_state=123,
                                 subsample=0.9101431135071837,
                                 validation_fraction=0.9964423942006735)

IBS: 0.22


In [231]:
# Saving the values to the dictionary 
test_cindex['GradientBoosting'] = c_index
test_ibs['GradientBoosting'] = ibs

### 8. ComponentwiseGradientBoostingSurvivalAnalysis

#### Train

In [232]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

In [233]:
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        subsample = trial.suggest_float("subsample", 0.1, 1)
        dropout_rate = trial.suggest_float("dropout_rate", 0.1, 1)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
        
        # Create and fit survival model 
        model = model_class(subsample=subsample,
                            dropout_rate=dropout_rate,
                            n_estimators=n_estimators,
                            learning_rate=learning_rate,
                            random_state=123)
                
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Transformation
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            pt = PowerTransformer(method='yeo-johnson')
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = pt.fit_transform(X_train_included)
                X_test_included_std = pt.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective


# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(ComponentwiseGradientBoostingSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best hyperparameters for C-index: \n", study_cindex.best_params)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(ComponentwiseGradientBoostingSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best hyperparameters for IBS: \n", study_ibs.best_params)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-14 15:58:53,333] A new study created in memory with name: no-name-25c9cbdd-ffab-4aeb-a664-e02699f00869


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.7402597402597403
Fold 2 C-index: 0.6473214285714286
Fold 3 C-index: 0.7230392156862745
Fold 4 C-index: 0.8185654008438819
Fold 5 C-index: 0.5985915492957746
[I 2024-04-14 15:58:54,206] Trial 0 finished with value: 0.70555546693142 and parameters: {'subsample': 0.7268222670380755, 'dropout_rate': 0.3575254014553415, 'n_estimators': 114, 'learning_rate': 0.05558016213920623}. Best is trial 0 with value: 0.70555546693142.
Fold 1 C-index: 0.7402597402597403
Fold 2 C-index: 0.6339285714285714
Fold 3 C-index: 0.7352941176470589
Fold 4 C-index: 0.8185654008438819
Fold 5 C-index: 0.5985915492957746
[I 2024-04-14 15:59:01,146] Trial 1 finished with value: 0.7053278758950055 and parameters: {'subsample': 0.7475220728070068, 'dropout_rate': 0.4807958141120149, 'n_estimators': 491, 'learning_rate': 0.06879814411990147}. Best is trial 0 with value: 0.70555546693142.
Fold 1 C-index: 0.7402597402597403
Fold 2 C-index: 0.6473214285714286
Fold 3 C-index: 0.7401960784313726
Fold 4 C-in

Fold 1 C-index: 0.7575757575757576
Fold 2 C-index: 0.6607142857142857
Fold 3 C-index: 0.7843137254901961
Fold 4 C-index: 0.8354430379746836
Fold 5 C-index: 0.5915492957746479
[I 2024-04-14 15:59:55,143] Trial 19 finished with value: 0.7259192205059142 and parameters: {'subsample': 0.16691848936207518, 'dropout_rate': 0.573860614797082, 'n_estimators': 299, 'learning_rate': 0.017941064631859}. Best is trial 14 with value: 0.7366584738240586.
Fold 1 C-index: 0.7489177489177489
Fold 2 C-index: 0.6428571428571429
Fold 3 C-index: 0.7573529411764706
Fold 4 C-index: 0.8312236286919831
Fold 5 C-index: 0.5868544600938967
[I 2024-04-14 15:59:59,279] Trial 20 finished with value: 0.7134411843474483 and parameters: {'subsample': 0.29019430370134297, 'dropout_rate': 0.7703393766216369, 'n_estimators': 412, 'learning_rate': 0.0016589351820883932}. Best is trial 14 with value: 0.7366584738240586.
Fold 1 C-index: 0.7705627705627706
Fold 2 C-index: 0.6741071428571429
Fold 3 C-index: 0.7843137254901961


Fold 1 C-index: 0.7619047619047619
Fold 2 C-index: 0.6607142857142857
Fold 3 C-index: 0.7794117647058824
Fold 4 C-index: 0.8354430379746836
Fold 5 C-index: 0.5915492957746479
[I 2024-04-14 16:00:58,116] Trial 38 finished with value: 0.7258046292148522 and parameters: {'subsample': 0.14807719394046315, 'dropout_rate': 0.6401103186758491, 'n_estimators': 242, 'learning_rate': 0.029594359565260984}. Best is trial 22 with value: 0.7367558310784401.
Fold 1 C-index: 0.7532467532467533
Fold 2 C-index: 0.65625
Fold 3 C-index: 0.7671568627450981
Fold 4 C-index: 0.8354430379746836
Fold 5 C-index: 0.5821596244131455
[I 2024-04-14 16:01:03,590] Trial 39 finished with value: 0.7188512556759361 and parameters: {'subsample': 0.2250121153664736, 'dropout_rate': 0.43471642740612565, 'n_estimators': 423, 'learning_rate': 0.006059367484731356}. Best is trial 22 with value: 0.7367558310784401.
Fold 1 C-index: 0.7748917748917749
Fold 2 C-index: 0.6785714285714286
Fold 3 C-index: 0.7843137254901961
Fold 4 C

...

Fold 3 IBS: 0.17561521038419572
Fold 4 IBS: 0.19617937926603082
Fold 5 IBS: 0.22030982851403455
[I 2024-04-14 16:04:31,033] Trial 19 finished with value: 0.19534422467932158 and parameters: {'subsample': 0.10013306718236009, 'dropout_rate': 0.9930703343219778, 'n_estimators': 51, 'learning_rate': 0.04096883532946153}. Best is trial 19 with value: 0.19534422467932158.
Fold 1 IBS: 0.1934568269543391
Fold 2 IBS: 0.19715811856127513
Fold 3 IBS: 0.17946469994393788
Fold 4 IBS: 0.20156135236511127
Fold 5 IBS: 0.21995157160040674
[I 2024-04-14 16:04:31,454] Trial 20 finished with value: 0.19831851388501404 and parameters: {'subsample': 0.1435040576337751, 'dropout_rate': 0.7933084651006226, 'n_estimators': 43, 'learning_rate': 0.04127909023805986}. Best is trial 19 with value: 0.19534422467932158.
Fold 1 IBS: 0.19629608820460723
Fold 2 IBS: 0.20116625589103407
Fold 3 IBS: 0.18307602213830743
Fold 4 IBS: 0.20415973589751046
Fold 5 IBS: 0.21825141892231678
[I 2024-04-14 16:04:31,748] Trial 21 f

Fold 3 IBS: 0.21726677705287487
Fold 4 IBS: 0.3067540059676042
Fold 5 IBS: 0.3088785718394419
[I 2024-04-14 16:04:47,978] Trial 38 finished with value: 0.2516627447151949 and parameters: {'subsample': 0.1954790971490823, 'dropout_rate': 0.8663362825803563, 'n_estimators': 197, 'learning_rate': 0.07288612365177065}. Best is trial 32 with value: 0.19011279663103947.
Fold 1 IBS: 0.17384033345948605
Fold 2 IBS: 0.19117022812858242
Fold 3 IBS: 0.1658112441781456
Fold 4 IBS: 0.19062854819614738
Fold 5 IBS: 0.2531160401238152
[I 2024-04-14 16:04:48,713] Trial 39 finished with value: 0.19491327881723533 and parameters: {'subsample': 0.2485509072011421, 'dropout_rate': 0.2538651184588351, 'n_estimators': 81, 'learning_rate': 0.05575307634287385}. Best is trial 32 with value: 0.19011279663103947.
Fold 1 IBS: 0.17198132816848963
Fold 2 IBS: 0.2379402139667447
Fold 3 IBS: 0.20755181076017126
Fold 4 IBS: 0.28872031988166974
Fold 5 IBS: 0.30715041622153244
[I 2024-04-14 16:04:49,768] Trial 40 finish

Fold 3 IBS: 0.1659808217178906
Fold 4 IBS: 0.1885288450582718
Fold 5 IBS: 0.25937603443602375
[I 2024-04-14 16:05:12,134] Trial 57 finished with value: 0.19367460739280726 and parameters: {'subsample': 0.10158446690685444, 'dropout_rate': 0.17532364403171014, 'n_estimators': 133, 'learning_rate': 0.048923651208961784}. Best is trial 51 with value: 0.18758692413271766.
Fold 1 IBS: 0.18163819459855188
Fold 2 IBS: 0.18938416944320802
Fold 3 IBS: 0.17070754202477906
Fold 4 IBS: 0.19532813835815624
Fold 5 IBS: 0.22827735864019225
[I 2024-04-14 16:05:12,670] Trial 58 finished with value: 0.1930670806129775 and parameters: {'subsample': 0.7591484265070766, 'dropout_rate': 0.10710740658854133, 'n_estimators': 53, 'learning_rate': 0.06216375518515009}. Best is trial 51 with value: 0.18758692413271766.
Fold 1 IBS: 0.18818122279496977
Fold 2 IBS: 0.1924822958080667
Fold 3 IBS: 0.1745201537844417
Fold 4 IBS: 0.19623717827044204
Fold 5 IBS: 0.2253054718123723
[I 2024-04-14 16:05:13,025] Trial 59 fi

Fold 4 IBS: 0.19225152703112275
Fold 5 IBS: 0.24483129098310055
[I 2024-04-14 16:05:37,508] Trial 76 finished with value: 0.19507245960980352 and parameters: {'subsample': 0.7158013476720768, 'dropout_rate': 0.13019725043087785, 'n_estimators': 75, 'learning_rate': 0.06127673455407747}. Best is trial 51 with value: 0.18758692413271766.
Fold 1 IBS: 0.20048691718406164
Fold 2 IBS: 0.2073649027207117
Fold 3 IBS: 0.1897554383249612
Fold 4 IBS: 0.21150446590995753
Fold 5 IBS: 0.21634700092017156
[I 2024-04-14 16:05:37,772] Trial 77 finished with value: 0.20509174501197275 and parameters: {'subsample': 0.1253062349895917, 'dropout_rate': 0.9060890066502643, 'n_estimators': 16, 'learning_rate': 0.05673526298023662}. Best is trial 51 with value: 0.18758692413271766.
Fold 1 IBS: 0.16495807583311792
Fold 2 IBS: 0.19756648497686738
Fold 3 IBS: 0.17396450451531964
Fold 4 IBS: 0.1719002916858016
Fold 5 IBS: 0.2648373708929835
[I 2024-04-14 16:05:38,640] Trial 78 finished with value: 0.1946453455808

Fold 4 IBS: 0.19194585623129345
Fold 5 IBS: 0.2725075187014806
[I 2024-04-14 16:06:03,064] Trial 95 finished with value: 0.1998963598377821 and parameters: {'subsample': 0.15581855729613242, 'dropout_rate': 0.4970595697042566, 'n_estimators': 214, 'learning_rate': 0.029994587608032502}. Best is trial 51 with value: 0.18758692413271766.
Fold 1 IBS: 0.1684853889941405
Fold 2 IBS: 0.1898378515109147
Fold 3 IBS: 0.16348142475190114
Fold 4 IBS: 0.17300211213188432
Fold 5 IBS: 0.2517731530441877
[I 2024-04-14 16:06:04,389] Trial 96 finished with value: 0.18931598608660566 and parameters: {'subsample': 0.10088444704400297, 'dropout_rate': 0.4137282240267256, 'n_estimators': 179, 'learning_rate': 0.0269141539102375}. Best is trial 51 with value: 0.18758692413271766.
Fold 1 IBS: 0.16741374104074316
Fold 2 IBS: 0.1947282103384171
Fold 3 IBS: 0.1660941001615167
Fold 4 IBS: 0.1824650024349122
Fold 5 IBS: 0.25946103645491997
[I 2024-04-14 16:06:06,582] Trial 97 finished with value: 0.19403241808610

In [234]:
train_cindex['ComponentwiseGradientBoosting'] = np.round(study_cindex.best_value, 3)
train_ibs['ComponentwiseGradientBoosting'] = np.round(study_ibs.best_value, 3)

In [235]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.739
train_ibs:  0.188


#### Test

In [236]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [237]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(ComponentwiseGradientBoostingSurvivalAnalysis, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(ComponentwiseGradientBoostingSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

ComponentwiseGradientBoostingSurvivalAnalysis(dropout_rate=0.6911773987018819,
                                              learning_rate=0.0038777019677429035,
                                              n_estimators=313,
                                              random_state=123,
                                              subsample=0.10055409799174729)

C-index score: 0.514


ComponentwiseGradientBoostingSurvivalAnalysis(dropout_rate=0.10045382034687865,
                                              learning_rate=0.05674565223918392,
                                              n_estimators=79, random_state=123,
                                              subsample=0.143043777223833)

IBS: 0.238


In [238]:
# Saving the values to the dictionary 
test_cindex['ComponentwiseGradientBoosting'] = c_index
test_ibs['ComponentwiseGradientBoosting'] = ibs

# Results

In [239]:
df_train_cindex = pd.DataFrame(train_cindex, index=['C-index']).transpose().sort_values(by='C-index', ascending=False)
df_train_cindex['rank'] = df_train_cindex['C-index'].rank(ascending=False)
df_train_cindex 

,C-index,rank
Randomsurvivalforest,0.855,1.0
ExtraSurvivalTrees,0.848,2.0
CoxPH,0.763,3.0
CoxElastic,0.761,4.0
CoxLasso,0.760,5.0
GradientBoosting,0.740,6.0
ComponentwiseGradientBoosting,0.739,7.0
CoxRidge,0.688,8.0


In [240]:
df_train_ibs = pd.DataFrame(train_ibs, index=['IBS']).transpose().sort_values(by='IBS', ascending=True)
df_train_ibs['rank'] = df_train_ibs['IBS'].rank(ascending=True)
df_train_ibs

,IBS,rank
CoxPH,0.180,2.0
CoxElastic,0.180,2.0
ExtraSurvivalTrees,0.180,2.0
CoxLasso,0.181,4.0
Randomsurvivalforest,0.182,5.0
ComponentwiseGradientBoosting,0.188,6.0
GradientBoosting,0.215,7.0
CoxRidge,0.217,8.0


In [241]:
df_test_cindex = pd.DataFrame(test_cindex, index=['C-index']).transpose().sort_values(by='C-index', ascending=False)
df_test_cindex['rank'] = df_test_cindex['C-index'].rank(ascending=False)
df_test_cindex 

,C-index,rank
Randomsurvivalforest,0.643,1.0
GradientBoosting,0.612,2.0
ExtraSurvivalTrees,0.610,3.0
CoxLasso,0.577,4.5
CoxElastic,0.577,4.5
CoxPH,0.572,6.0
CoxRidge,0.534,7.0
ComponentwiseGradientBoosting,0.514,8.0


In [242]:
df_test_ibs = pd.DataFrame(test_ibs, index=['IBS']).transpose().sort_values(by='IBS', ascending=True)
df_test_ibs['rank'] = df_test_ibs['IBS'].rank(ascending=True)
df_test_ibs 

,IBS,rank
Randomsurvivalforest,0.206,1.0
GradientBoosting,0.220,2.0
CoxRidge,0.221,3.0
ExtraSurvivalTrees,0.230,4.0
ComponentwiseGradientBoosting,0.238,5.0
CoxLasso,0.269,6.5
CoxElastic,0.269,6.5
CoxPH,0.274,8.0


In [243]:
# Renaming the column "index" to "model" 
df_train_cindex = df_train_cindex.reset_index().rename(columns={"index": "model"})
df_train_ibs = df_train_ibs.reset_index().rename(columns={"index": "model"})
df_test_cindex = df_test_cindex.reset_index().rename(columns={"index": "model"})
df_test_ibs = df_test_ibs.reset_index().rename(columns={"index": "model"})

# Save the files 
dfs = [df_train_cindex, df_train_ibs, df_test_cindex, df_test_ibs]  # List of your DataFrames
file_path = 'path_to_your_folder/'  # Folder path where you want to save the files

# List of corresponding file names
file_names = ['train_cindex.csv', 'train_ibs.csv', 'test_cindex.csv', 'test_ibs.csv']


dfs = [df_train_cindex, df_train_ibs, df_test_cindex, df_test_ibs]  # List of your DataFrames
file_path = '/Users/minjeongcheon/Desktop/results_thesis/d1/os/yeojohnson/rent/'  # Folder path where you want to save the files

# List of corresponding file names
file_names = ['train_cindex.csv', 'train_ibs.csv', 'test_cindex.csv', 'test_ibs.csv']

# Modify the file names to match the desired format
modified_file_names = ['d1_os_yeojohnson_rent_' + file_name for file_name in file_names]

# Loop through each DataFrame and save them with corresponding modified file names
for df, modified_file_name in zip(dfs, modified_file_names):
    file_path_name = file_path + modified_file_name  # Construct the full file path
    df.to_csv(file_path_name, index=False)  # Save the DataFrame to CSV file


In [244]:
from datetime import date
today = date.today()
print("Date: ", today)

Date:  2024-04-14
